In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pmhclib.components.pmhc import TCRpMHCII

import plotly.io as pio

pio.templates.default = "plotly_white"

In [ ]:
library_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
library_df.sort_values(by=["peptide_len", "left_pfr_len"])

In [ ]:
templates = dict()
overflows = dict()

max_left_pfr_len = library_df.left_pfr_len.max()
max_right_pfr_len = library_df.right_pfr_len.max()

for pdb_id, record in library_df.iterrows():
    templates[pdb_id] = TCRpMHCII(
        f"../data/mhcii_tcr_templates/{pdb_id}.pdb", id=pdb_id,
        mhc_allotype=record.allotype, anchors=[record[f"anch_{i + 1}"] for i in range(4)]
    )
    overflows[pdb_id] = (
        max_left_pfr_len - record.left_pfr_len,
        max_right_pfr_len - record.right_pfr_len
    )
    print(record.anchors)

In [ ]:
def fill_peptide_vector(vec: np.ndarray, pdb_id: str) -> np.ndarray:

    (left_overflow, right_overflow) = overflows[pdb_id]
    if left_overflow > 0:
        vec = np.concatenate([np.full(left_overflow, None), vec])
    if right_overflow > 0:
        vec = np.concatenate([vec, np.full(right_overflow, None)])

    return vec

In [ ]:
sasa_X = list()

for pdb_id, template in templates.items():
    sasa_X.append(fill_peptide_vector(template.peptide_sasa(), pdb_id))

sasa_X = np.array(sasa_X)

In [ ]:
fig = px.imshow(sasa_X.T)
fig.update_xaxes(
    title="PDB ID",
    tickmode="array",
    tickvals=list(range(sasa_X.shape[0])),
    ticktext=library_df.index.to_list()
)
fig.update_yaxes(
    title="Adjusted residue index",
    tickmode="array",
    tickvals=list(range(sasa_X.shape[1])),
    ticktext=list(range(1, sasa_X.shape[1] + 1))
)
fig.update_layout(plot_bgcolor="white")
fig.show()

In [ ]:
contact_X = list()

for pdb_id, template in templates.items():
    contact_X.append(fill_peptide_vector(template.get_peptide_tcr_contacts(distance=3.0), pdb_id))

contact_X = np.array(contact_X)

In [ ]:
fig = px.imshow(contact_X.T)
fig.update_xaxes(
    title="PDB ID",
    tickmode="array",
    tickvals=list(range(contact_X.shape[0])),
    ticktext=library_df.index.to_list()
)
fig.update_yaxes(
    title="Adjusted residue index",
    tickmode="array",
    tickvals=list(range(contact_X.shape[1])),
    ticktext=list(range(1, contact_X.shape[1] + 1))
)
fig.update_layout(plot_bgcolor="white")
fig.show()

In [ ]:
crossing_angles = dict()
incident_angles = dict()

for pdb_id, template in templates.items():
    [crossing_angle, incident_angle] = template.compute_docking_angles()
    crossing_angles[pdb_id] = crossing_angle
    incident_angles[pdb_id] = incident_angle

library_df["crossing_angle"] = library_df.index.map(crossing_angles)
library_df["incident_angle"] = library_df.index.map(incident_angles)

X = np.array(library_df[["crossing_angle", "incident_angle"]].values)

In [ ]:
fig = px.histogram(x=X[:, 0])
fig.update_xaxes(title="Crossing angle (twist)", range=(0, 180))
fig.update_traces(xbins={"size": 2})
fig.show()

fig = px.histogram(x=X[:, 1])
fig.update_xaxes(title="Incident angle (tilt)", range=(0, 90))
fig.update_traces(xbins={"size": 1})
fig.show()

fig = px.scatter(x=X[:, 0], y=X[:, 1], hover_name=library_df.index.to_list())
fig.update_xaxes(range=(0, 180))
fig.update_yaxes(range=(0, 90))
fig.show()

In [ ]:
pierce_angles = {
    "1D9K": [53.4, 6.9],
    "1FYT": [47.2, 2.8],
    "1J8H": [48.9, 1.0],
    "1U3H": [43.7, 0.7],
    "1YMM": [83.5, 18.0],
    "1ZGL": [40.3, 8.7],
    "2IAM": [44.4, 15.6],
    "2IAN": [44.0, 15.5],
    "2PXY": [49.1, 3.7],
    "2WBJ": [80.3, 19.6],
    "2Z31": [48.9, 3.1],
    "3C5Z": [45.9, 11.0],
    "3C60": [53.1, 6.9],
    "3C6L": [44.4, 2.4],
    "3MBE": [48.2, 15.1],
    "3O6F": [36.4, 6.0],
    "3PL6": [25.3, 14.3],
    "3QIB": [36.9, 11.8],
    "3QIU": [39.4, 12.1],
    "3QIW": [40.4, 11.9],
    "3RDT": [41.4, 7.2],
    "3T0E": [37.5, 7.9],
    "4E41": [46.6, 17.3],
    "4GG6": [39.9, 6.8],
    "4GRL": [26.1, 14.6],
    "4H1L": [56.0, 11.6],
    "4MAY": [25.5, 14.3],
    "4OZF": [46.2, 13.2],
    "4OZG": [36.1, 6.4],
    "4OZH": [36.2, 5.8],
    "4OZI": [46.0, 21.9],
    "4P23": [41.1, 6.7],
    "4P2O": [40.1, 12.7],
    "4P2Q": [32.2, 15.9],
    "4P2R": [32.3, 15.5],
    "4P46": [41.7, 7.4],
    "4P4K": [46.6, 24.8],
    "4P5T": [45.3, 7.8],
    "4Y19": [136.9, 43.2],
    "4Y1A": [137.6, 44.8],
    "4Z7U": [37.9, 2.5],
    "4Z7V": [39.6, 5.1],
    "4Z7W": [46.1, 9.0],
    "5KS9": [39.5, 8.5],
    "5KSA": [42.1, 5.5],
    "5KSB": [33.8, 11.8],
    "6BGA": [36.6, 13.8],
    "6CQL": [34.3, 5.5],
    "6CQN": [33.7, 5.0],
    "6CQQ": [35.1, 5.3],
    "6CQR": [31.4, 5.6],
    "6DFS": [37.7, 16.7],
    "6DFW": [49.4, 17.3],
    "6DFX": [35.8, 1.7],
    "6MKD": [42.0, 15.3],
    "6MKR": [41.6, 15.2],
    "6MNG": [46.6, 13.8],
    "6MNM": [43.5, 14.4],
    "6MNN": [46.2, 13.5],
    "6MNO": [44.5, 12.8],
    "6PX6": [30.8, 4.0],
    "6PY2": [53.3, 14.6],
    "6R0E": [48.6, 2.9],
    "6U3N": [34.6, 12.2],
    "6U3O": [48.2, 8.1],
    "6V0Y": [24.0, 16.7],
    "6V13": [25.9, 16.4],
    "6V15": [26.0, 16.1],
    "6V18": [24.0, 16.6],
    "6V19": [24.5, 16.0],
    "6V1A": [25.5, 16.8],
    "6XC9": [32.3, 9.9],
    "6XCO": [35.5, 7.0],
    "6XCP": [34.9, 5.3],
    "7RDV": [42.6, 11.9],
    "7SG0": [40.5, 10.4],
    "7SG1": [37.5, 14.7],
    "7SG2": [37.1, 14.3],
    "7T2B": [36.9, 6.5],
    "7T2C": [52.4, 13.0],
    "7T2D": [54.5, 12.5],
    "7Z50": [47.0, 4.1],
    "8PJG": [48.7, 2.9],
    "8TRL": [35.6, 3.7],
    "8TRQ": [32.0, 8.7],
    "8TRR": [38.8, 9.5],
    "8VCX": [33.0, 4.3],
    "8VCY": [33.7, 3.7],
    "8VD0": [None, None],
    "8VD2": [40.3, 5.2]
}

pierce_crossing_angles = library_df.index.map({k: v[0] for k, v in pierce_angles.items()})
pierce_incident_angles = library_df.index.map({k: v[1] for k, v in pierce_angles.items()})

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=library_df.index, y=library_df.crossing_angle.values, name="ours"))
fig.add_trace(go.Scatter(x=library_df.index, y=pierce_crossing_angles.values, name="TCR3d"))
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=library_df.index, y=library_df.incident_angle.values, name="ours"))
fig.add_trace(go.Scatter(x=library_df.index, y=pierce_incident_angles.values, name="TCR3d"))
fig.show()

In [ ]:
cdr3_a = dict()
cdr3_b = dict()

for pdb_id, template in templates.items():
    cdr3_a[pdb_id] = ";".join([str(x) for x in template.tcr.alpha_chain.cdr_loops[2]])
    cdr3_b[pdb_id] = ";".join([str(x) for x in template.tcr.alpha_chain.cdr_loops[2]])

library_df["cdr3_a"] = library_df.index.map(cdr3_a)
library_df["cdr3_b"] = library_df.index.map(cdr3_b)

library_df

In [ ]:
bc_seqs = library_df.apply(lambda row: [row.peptide_seq[i - 1] for i in range(row.anch_1, row.anch_4 + 1)], axis=1)
count_mat = pd.DataFrame(bc_seqs.values.tolist()).apply(pd.Series.value_counts).fillna(0).T.astype(int)
p_mat = count_mat.div(count_mat.sum(axis=1), axis=0)

fig = px.imshow(p_mat)
p_mat = np.round(p_mat, 2)
text = np.where(p_mat == 0, "", p_mat)
fig.update_yaxes(title="Binding core residue", tickmode="array", tickvals=list(range(p_mat.shape[0])), ticktext=list(range(1, p_mat.shape[0] + 1)))
fig.update_xaxes(title="Amino acid")
fig.update_traces(
    text=text,
    texttemplate="%{text}"
)
fig.show()

In [ ]:
from tqdm import tqdm

from pymol import cmd

cmd.loadall("../data/mhcii_tcr_templates/*.pdb", "templates")
cmd.create("ref", "1D9K")

for pdb_id in tqdm(library_df.index.to_list()):
    cmd.align(f"{pdb_id} & chain M+N", "ref & chain M+N")
    cmd.color("deeppurple", "chain M")
    cmd.color("lightpink", "chain N")
    cmd.color("orange", "chain P")
    cmd.color("deepblue", "chain A")
    cmd.color("lightblue", "chain B")

cmd.create("anchors", " or ".join([f"({pdb_id} & chain P & (resi {row.anch_1} or resi {row.anch_2} or resi {row.anch_3} or resi {row.anch_4}))" for pdb_id, row in library_df.iterrows()]))

cmd.save("library.pse")
cmd.delete("all")